# CBoE · SDH 자막 검수 화면 (Colab)

1. `cboe-web.zip` 올리기 → 2. 작업 폴더(JSON)·영상 올리기 → 3. 검수 화면 열기

파이프라인이 만든 `items.json` · `dialogue.json` · `cuts.json` · `project.json` (+ 있으면 `review.json`)을 그대로 쓰면 돼요.

In [ ]:
import os, zipfile
from google.colab import files
if not os.path.exists('/content/cboe-web/index.html'):
    for name in files.upload():
        if name.endswith('.zip'): zipfile.ZipFile(name).extractall('/content')
# 드라이브: from google.colab import drive; drive.mount('/content/drive'); !cp -r /content/drive/MyDrive/cboe-web /content/
os.chdir('/content/cboe-web'); print(sorted(os.listdir('.')))

## 2. 작업 폴더 · 영상 올리기
JSON 파일들과 영상을 한꺼번에 골라도 돼요. 아무것도 안 올리면 샘플(`work/sample`)로 열려요. 영상이 크면 드라이브에서 `media/`로 복사하세요.

In [ ]:
import shutil
WORK, VIDEO, SUBS = 'work/sample', None, None
up = files.upload()
if up:
    os.makedirs('work/mine', exist_ok=True)
for name in up:
    ext = name.lower().rsplit('.', 1)[-1]
    if ext in ('mp4','webm','mov','m4v'): shutil.move(name, f'media/{name}'); VIDEO = f'media/{name}'
    elif ext == 'json': shutil.move(name, f'work/mine/{name}'); WORK = 'work/mine'
    elif ext in ('srt','vtt'): shutil.move(name, f'work/{name}'); SUBS = f'work/{name}'
print('작업:', WORK, '| 영상:', VIDEO, '| 자막:', SUBS)

## 3. 검수 화면 열기
아래에 바로 뜨고, 링크로 새 탭에서 크게 볼 수 있어요. 결정은 **review.json 저장** 버튼으로 내려받아 파이프라인 작업 폴더에 넣으면 돼요.

In [ ]:
import sys, urllib.parse
os.environ['CBOE_QUIET'] = '1'; sys.path.insert(0, '/content/cboe-web')
import serve
try: serve.start(8000, background=True)
except OSError: pass
q = {k: v for k, v in (('subs', SUBS),) if v} if SUBS and WORK == 'work/sample' else {'work': WORK}
if VIDEO: q['video'] = VIDEO
path = '/studio.html?' + urllib.parse.urlencode(q)
from google.colab import output
output.serve_kernel_port_as_window(8000, path=path)
output.serve_kernel_port_as_iframe(8000, path=path, height=900)